In [8]:
%pip install numpy scipy scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

DATA_PATH = "seq_dataset_10dim_v7.npz"
SCALER_PATH = "scaler_10dim_v7.json"

d = np.load(DATA_PATH, allow_pickle=True)
features = d['features'].tolist()

# Read train-set scaler parameters used when building the 10D dataset
with open(SCALER_PATH, "r", encoding="utf-8") as f:
    scaler_cfg = json.load(f)

# Make sure the scaler and dataset use exactly the same feature order
scaler_features = scaler_cfg["features"]
if features != scaler_features:
    raise ValueError(
        "Feature order mismatch between dataset and scaler.\n"
        f"dataset: {features}\n"
        f"scaler:  {scaler_features}"
    )

scaler_mean = np.asarray(scaler_cfg["mean"], dtype=float)
scaler_std = np.asarray(scaler_cfg["std"], dtype=float)

features


['SOH',
 'SOC10_R0',
 'SOC50_R0',
 'SOC90_R0',
 'SOC10_R1',
 'SOC50_R1',
 'SOC90_R1',
 'SOC10_R2',
 'SOC50_R2',
 'SOC90_R2']

In [10]:
# Dataset structure
for split in ['train', 'val', 'test']:
    print(f'\n[{split}]')
    print('X:', d[f'{split}_X'].shape)
    print('Y:', d[f'{split}_Y'].shape)
    print('unique batteries:', len(np.unique(d[f'{split}_bid'])))

print('\nFeature order:')
for i, f in enumerate(features):
    print(i, f)



[train]
X: (4581, 30, 10)
Y: (4581, 29, 10)
unique batteries: 212

[val]
X: (45, 30, 10)
Y: (45, 29, 10)
unique batteries: 45

[test]
X: (45, 30, 10)
Y: (45, 29, 10)
unique batteries: 45

Feature order:
0 SOH
1 SOC10_R0
2 SOC50_R0
3 SOC90_R0
4 SOC10_R1
5 SOC50_R1
6 SOC90_R1
7 SOC10_R2
8 SOC50_R2
9 SOC90_R2


In [ ]:
# X：模型输入序列
# Y：未来预测目标
# X_fmask：某个特征是否真实存在，用于处理原始 NaN
# X_tmask：这个时间步是不是 padding
# Y_mask：预测目标是否有效
# Y_shape：决定 prediction array的大小

def persistence_predict(X, X_fmask, X_tmask, y_shape):
    n, T_future, d_feat = y_shape
    pred = np.full((n, T_future, d_feat), np.nan, dtype=np.float32)
    pred_available = np.zeros((n, T_future, d_feat), dtype=bool)

    for i in range(n):
        obs_idx = np.where(X_tmask[i])[0]
        if len(obs_idx) == 0:
            continue

        for j in range(d_feat):
            valid_obs_idx = obs_idx[X_fmask[i, obs_idx, j]]
            if len(valid_obs_idx) == 0:
                continue

            last_value = X[i, valid_obs_idx[-1], j]
            pred[i, :, j] = last_value
            pred_available[i, :, j] = True

    return pred, pred_available


def masked_metrics(y_true, y_pred, mask):
    yt = y_true[mask].astype(float)
    yp = y_pred[mask].astype(float)

    if len(yt) == 0:
        return {'N': 0, 'MAE': np.nan, 'RMSE': np.nan, 'R2': np.nan}

    mae = np.mean(np.abs(yp - yt))
    rmse = np.sqrt(np.mean((yp - yt) ** 2))
    r2 = r2_score(yt, yp) if len(yt) >= 2 else np.nan
    return {'N': len(yt), 'MAE': mae, 'RMSE': rmse, 'R2': r2}


# standardize
def inverse_transform(arr):
    out = arr.astype(float).copy()
    out[..., 0] = arr[..., 0] * scaler_std[0] + scaler_mean[0]
    out[..., 1:] = np.expm1(arr[..., 1:] * scaler_std[1:] + scaler_mean[1:])
    return out


In [ ]:
# pred_raw：物理尺度 prediction
# eval_mask：有效评价位置
# pd.DataFrame(rows)：10 个 individual target 的 metrics table

def evaluate_persistence(split):
    X = d[f'{split}_X']
    X_fmask = d[f'{split}_X_fmask']
    X_tmask = d[f'{split}_X_tmask']
    Y = d[f'{split}_Y']
    Y_mask = d[f'{split}_Y_mask']

    pred_scaled, pred_available = persistence_predict(X, X_fmask, X_tmask, Y.shape)
    eval_mask = Y_mask & pred_available

    Y_raw = inverse_transform(Y)
    pred_raw = inverse_transform(pred_scaled)

    rows = []
    for j, f in enumerate(features):
        m = masked_metrics(Y_raw[:, :, j], pred_raw[:, :, j], eval_mask[:, :, j])
        rows.append({'Target': f, **m})

    return pred_raw, eval_mask, pd.DataFrame(rows)

val_pred, val_eval_mask, val_metrics = evaluate_persistence('val')
test_pred, test_eval_mask, test_metrics = evaluate_persistence('test')

print('Validation — physical scale')
display(val_metrics)
print('Test — physical scale')
display(test_metrics)


Validation — physical scale

,Target,N,MAE,RMSE,R2
0,SOH,662,8.894411,10.711799,-0.736678
1,SOC10_R0,659,5.945571,7.687646,-0.800582
2,SOC50_R0,658,5.996127,7.869253,-0.656457
3,SOC90_R0,590,6.708938,8.352437,0.134774
4,SOC10_R1,658,3.087154,7.529150,0.155266
5,SOC50_R1,656,2.419928,4.415664,0.089578
6,SOC90_R1,467,8.516142,11.933765,0.181480
7,SOC10_R2,634,1.020748,1.798418,0.258070
8,SOC50_R2,630,0.531462,0.767036,-0.365319
9,SOC90_R2,382,0.945817,1.160112,-0.868590


Test — physical scale


,Target,N,MAE,RMSE,R2
0,SOH,622,8.554502,10.191090,-0.693323
1,SOC10_R0,614,5.479086,6.990706,-0.429166
2,SOC50_R0,616,5.618134,7.211927,-0.499639
3,SOC90_R0,558,6.130574,7.579691,-0.425890
4,SOC10_R1,613,2.118637,4.171457,0.291989
5,SOC50_R1,616,2.048676,3.363377,-0.076801
6,SOC90_R1,450,7.372729,9.796295,-0.373769
7,SOC10_R2,603,0.852890,1.518443,0.242214
8,SOC50_R2,599,0.453898,0.618162,-0.294852
9,SOC90_R2,379,0.986368,1.160336,-1.160848


In [13]:
# Four-task summary in physical scale
task_map = {
    'SOH': [0],
    'R0': [1, 2, 3],
    'R1': [4, 5, 6],
    'R2': [7, 8, 9],
}

def task_summary(split, pred, eval_mask):
    Y_raw = inverse_transform(d[f'{split}_Y'])
    rows = []
    for task, idx in task_map.items():
        m = masked_metrics(Y_raw[:, :, idx], pred[:, :, idx], eval_mask[:, :, idx])
        rows.append({'Task': task, **m})
    return pd.DataFrame(rows)

print('Validation task summary — physical scale')
display(task_summary('val', val_pred, val_eval_mask))
print('Test task summary — physical scale')
display(task_summary('test', test_pred, test_eval_mask))


Validation task summary — physical scale


,Task,N,MAE,RMSE,R2
0,SOH,662,8.894411,10.711799,-0.736678
1,R0,1907,6.199191,7.960768,-0.205720
2,R1,1781,4.264940,8.091254,0.349403
3,R2,1646,0.816086,1.335406,0.776038


Test task summary — physical scale


,Task,N,MAE,RMSE,R2
0,SOH,622,8.554502,10.191090,-0.693323
1,R0,1788,5.730307,7.254696,-0.264184
2,R1,1679,3.501153,6.018660,0.249002
3,R2,1581,0.733720,1.160571,0.811033


In [14]:
# Check the current fixed observation-window setting in val/test
for split in ['val', 'test']:
    obs = d[f'{split}_X_tmask'].sum(axis=1)
    future = d[f'{split}_Y_mask'].any(axis=2).sum(axis=1)
    total = obs + future
    ratio = obs / total
    print(f'{split}: mean observed fraction = {ratio.mean():.4f}, min = {ratio.min():.4f}, max = {ratio.max():.4f}')


val: mean observed fraction = 0.4004, min = 0.3333, max = 0.4211
test: mean observed fraction = 0.4050, min = 0.3810, max = 0.4286
